# TCN pool model
Global TCN (Temporal Convolutional Network) trained on pooled data. Same features as LSTM: return lags, volume lag, RSI, MACD; **target = next 7 log returns**; compound to price via **p0 * exp(cumsum(log_returns))**. Uses Keras Conv1D with causal padding and dilation. Same 60-day test window and rolling backtest as other pool models.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    fetch_cnn_fear_greed_index,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TICKERS,
)

LAG_RETURNS = 5
RSI_PERIOD = 7
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9
SEQ_LEN = 30
TCN_EPOCHS = 40
TCN_FILTERS = 64
TCN_KERNEL = 3
TCN_DILATIONS = [1, 2, 4, 8]

In [2]:
def _rsi(series: pd.Series, period: int) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / np.where(avg_loss != 0, avg_loss, 1e-10)
    return 100 - (100 / (1 + rs))


def build_feature_df(grp: pd.DataFrame, for_prediction: bool = False):
    """Same as LSTM: return lags, volume lag, RSI, MACD. Target = next 7 **log returns**. If for_prediction=True, only drop rows where features are NaN (keep last row for prediction)."""
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    for i in range(1, LAG_RETURNS + 1):
        df[f"ret_lag_{i}"] = df["log_return"].shift(i)
    if "volume" in df.columns:
        df["volume_lag_1"] = df["volume"].astype(float).shift(1)
    else:
        df["volume_lag_1"] = np.nan
    df["rsi"] = _rsi(df["close"], RSI_PERIOD)
    ema_fast = df["close"].ewm(span=MACD_FAST, adjust=False).mean()
    ema_slow = df["close"].ewm(span=MACD_SLOW, adjust=False).mean()
    df["macd_line"] = ema_fast - ema_slow
    df["macd_signal"] = df["macd_line"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    if "vix" in df.columns:
        df["vix_lag_1"] = df["vix"].astype(float).shift(1)
    else:
        df["vix_lag_1"] = np.nan
    df["month"] = pd.to_datetime(df["timestamp"]).dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    if "fear_greed" not in df.columns:
        df["fear_greed"] = 50.0
    else:
        df["fear_greed"] = df["fear_greed"].fillna(50.0)
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["log_return"].shift(-h)
    feature_cols = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)] + [
        "volume_lag_1", "rsi", "macd_line", "macd_signal"
    ]
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "return", "log_return"] + feature_cols + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    for col in feature_cols:
        if col in out.columns and out[col].isna().all():
            out[col] = 0.0
    if for_prediction:
        out = out.dropna(subset=feature_cols)
    else:
        out = out.dropna()
    return out, feature_cols, target_cols


def build_sequences(X: np.ndarray, y: np.ndarray, seq_len: int):
    n = len(X)
    if n < seq_len + 1:
        return None, None
    X_seq, y_seq = [], []
    for i in range(seq_len, n):
        X_seq.append(X[i - seq_len : i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

In [3]:
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv1D, Dense, Lambda
    HAS_TCN = True
except Exception:
    HAS_TCN = False

def build_tcn(seq_len: int, n_feat: int, horizon: int, filters=64, kernel_size=3, dilations=None):
    if dilations is None:
        dilations = [1, 2, 4, 8]
    model = Sequential()
    for i, d in enumerate(dilations):
        model.add(Conv1D(filters, kernel_size, dilation_rate=d, padding="causal", activation="relu",
                        input_shape=(seq_len, n_feat) if i == 0 else None))
    model.add(Conv1D(horizon, 1))
    model.add(Lambda(lambda x: x[:, -1, :]))
    model.compile(optimizer="adam", loss="mse")
    return model

def train_tcn(X_seq: np.ndarray, y_seq: np.ndarray, horizon: int, epochs=40, filters=64, kernel_size=3, dilations=None, batch_size=256):
    if not HAS_TCN or X_seq is None or len(X_seq) < 10:
        return None
    if dilations is None:
        dilations = [1, 2, 4, 8]
    seq_len, n_feat = X_seq.shape[1], X_seq.shape[2]
    model = build_tcn(seq_len, n_feat, horizon, filters=filters, kernel_size=kernel_size, dilations=dilations)
    model.fit(X_seq, y_seq, epochs=epochs, batch_size=batch_size, verbose=0)
    return model

In [4]:
stacked = load_pool_data(with_vix=True, with_volume=False, use_fixed=True)
stacked = stacked.sort_values(["symbol", "timestamp"]).reset_index(drop=True)
if "fear_greed" not in stacked.columns:
    fng = fetch_cnn_fear_greed_index(use_fixed=True)
    if not fng.empty:
        stacked = stacked.merge(fng.drop_duplicates("timestamp"), on="timestamp", how="left")
    stacked["fear_greed"] = stacked.get("fear_greed", 50.0).fillna(50.0)

def train_global_tcn(stacked: pd.DataFrame, horizon: int, config: dict = None):
    """Train one TCN on pooled data. If config is provided, use it (seq_len, epochs, filters, kernel_size, dilations); else use module defaults. Returns dict with model, scaler, feature_cols, seq_len."""
    if config is None:
        config = {}
    seq_len = config.get("seq_len", SEQ_LEN)
    epochs = config.get("epochs", TCN_EPOCHS)
    filters = config.get("filters", TCN_FILTERS)
    kernel_size = config.get("kernel_size", TCN_KERNEL)
    dilations = config.get("dilations", TCN_DILATIONS)
    batch_size = config.get("batch_size", 256)
    pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
    if pooled.empty or not HAS_TCN:
        return None
    feat_dfs = []
    for sym in pooled["symbol"].unique():
        grp = pooled[pooled["symbol"] == sym].copy()
        try:
            feat_df, feature_cols, target_cols = build_feature_df(grp)
        except Exception:
            continue
        if len(feat_df) < MIN_TRAIN_STACK + horizon or len(feat_df) < seq_len + horizon + 1:
            continue
        feat_dfs.append((feat_df, feature_cols, target_cols))
    if not feat_dfs:
        return None
    feature_cols = feat_dfs[0][1]
    target_cols = feat_dfs[0][2]
    X_list, y_list = [], []
    for feat_df, _, _ in feat_dfs:
        X_list.append(feat_df[feature_cols].values.astype(np.float32))
        y_list.append(feat_df[target_cols].values.astype(np.float32))
    X_all = np.vstack(X_list)
    y_all = np.vstack(y_list)
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X_all)
    X_seq_list, y_seq_list = [], []
    for feat_df, _, _ in feat_dfs:
        X_sym = scaler.transform(feat_df[feature_cols].values.astype(np.float32))
        y_sym = feat_df[target_cols].values.astype(np.float32)
        X_seq, y_seq = build_sequences(X_sym, y_sym, seq_len)
        if X_seq is not None and len(X_seq) >= 10:
            X_seq_list.append(X_seq)
            y_seq_list.append(y_seq)
    if not X_seq_list:
        return None
    X_seq_all = np.concatenate(X_seq_list, axis=0)
    y_seq_all = np.concatenate(y_seq_list, axis=0)
    tcn_model = train_tcn(X_seq_all, y_seq_all, horizon, epochs=epochs, filters=filters, kernel_size=kernel_size, dilations=dilations, batch_size=batch_size)
    if tcn_model is None:
        return None
    return {"model": tcn_model, "scaler": scaler, "feature_cols": feature_cols, "seq_len": seq_len}


def predict_tcn_global(context_df: pd.DataFrame, horizon: int, global_tcn: dict) -> list:
    """Predict 7 price steps using pre-trained global TCN."""
    if global_tcn is None:
        return []
    try:
        feat_df, feature_cols, _ = build_feature_df(context_df, for_prediction=True)
    except Exception:
        return []
    seq_len = global_tcn.get("seq_len", SEQ_LEN)
    if len(feat_df) < seq_len + 1:
        return []
    X = feat_df[feature_cols].values.astype(np.float32)
    X_s = global_tcn["scaler"].transform(X)
    last_idx = len(X_s) - 1
    if last_idx < seq_len:
        return []
    seq = X_s[last_idx - seq_len : last_idx]
    pred_log_returns = global_tcn["model"].predict(seq.reshape(1, seq_len, -1), verbose=0).ravel()
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.exp(np.cumsum(pred_log_returns))
    return [float(p) for p in prices[:horizon]]

def run_backtest(stacked: pd.DataFrame, global_tcn: dict, horizon: int):
    """Run rolling backtest; returns DataFrame with timestamp, y_true, y_pred, window_ix, symbol."""
    all_preds = []
    for sym in TICKERS:
        grp = stacked[stacked["symbol"] == sym].copy()
        if grp.empty:
            continue
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        prices = grp.set_index("timestamp")["close"].astype(float).dropna()
        n = len(prices)
        if n < TEST_SIZE + MIN_TRAIN_STACK:
            continue
        split_idx = n - TEST_SIZE
        test_index = prices.index[split_idx:]
        test_values = prices.values[split_idx:]
        start, window_ix = 0, 0
        while start + horizon <= TEST_SIZE:
            if len(grp.iloc[: split_idx + start]) < MIN_TRAIN_STACK:
                start += ROLLING_STEP
                window_ix += 1
                continue
            price_list = predict_tcn_global(grp.iloc[: split_idx + start], horizon, global_tcn)
            if not price_list or len(price_list) < horizon:
                start += ROLLING_STEP
                window_ix += 1
                continue
            for h in range(horizon):
                idx = start + h
                all_preds.append({"timestamp": test_index[idx], "y_true": float(test_values[idx]), "y_pred": float(price_list[h]), "window_ix": window_ix, "symbol": sym})
            window_ix += 1
            start += ROLLING_STEP
    return pd.DataFrame(all_preds) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])

BEST_TCN_PARAMS_PATH = ARTIFACTS_DIR / "best_tcn_params.parquet"

In [5]:
# Random search for TCN hyperparameters; saves best to parquet for reuse
import random
import ast

param_grid = {
    "seq_len": [14, 21, 30],
    "epochs": [30, 40, 50],
    "filters": [32, 64],
    "kernel_size": [3, 5],
    "dilations": [[1, 2, 4, 8], [1, 2, 4, 8, 16]],
}
n_iter = 10  # small search
np.random.seed(42)
random.seed(42)

best_mae = np.inf
best_config = None
results = []

for trial in range(n_iter):
    config = {
        "seq_len": random.choice(param_grid["seq_len"]),
        "epochs": random.choice(param_grid["epochs"]),
        "filters": random.choice(param_grid["filters"]),
        "kernel_size": random.choice(param_grid["kernel_size"]),
        "dilations": random.choice(param_grid["dilations"]).copy(),
    }
    try:
        tcn = train_global_tcn(stacked, FORECAST_HORIZON, config=config)
        if tcn is None:
            continue
        pred_df = run_backtest(stacked, tcn, FORECAST_HORIZON)
        if pred_df.empty:
            continue
        m = compute_metrics_averaged_over_windows(pred_df)
        mae = m["MAE"]
        results.append({**config, "overall_mae": mae})
        if mae < best_mae:
            best_mae = mae
            best_config = config.copy()
            print(f"Trial {trial + 1}: new best MAE = {mae:.4f} | {config}")
    except Exception as e:
        print(f"Trial {trial + 1} failed: {e}")

if best_config is not None:
    best_row = {
        "seq_len": best_config["seq_len"],
        "epochs": best_config["epochs"],
        "filters": best_config["filters"],
        "kernel_size": best_config["kernel_size"],
        "dilations_str": str(best_config["dilations"]),
        "overall_mae": best_mae,
    }
    pd.DataFrame([best_row]).to_parquet(BEST_TCN_PARAMS_PATH, index=False)
    print("Saved best params to", BEST_TCN_PARAMS_PATH, "| MAE =", best_mae)
else:
    print("No valid config found; best_tcn_params.parquet not updated.")

c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Trial 1: new best MAE = 6.6988 | {'seq_len': 30, 'epochs': 30, 'filters': 32, 'kernel_size': 5, 'dilations': [1, 2, 4, 8]}


c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Trial 3: new best MAE = 6.5916 | {'seq_len': 30, 'epochs': 30, 'filters': 32, 'kernel_size': 5, 'dilations': [1, 2, 4, 8]}


c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init

Trial 7: new best MAE = 6.4189 | {'seq_len': 21, 'epochs': 30, 'filters': 32, 'kernel_size': 3, 'dilations': [1, 2, 4, 8, 16]}


c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Trial 8: new best MAE = 6.4018 | {'seq_len': 21, 'epochs': 40, 'filters': 32, 'kernel_size': 5, 'dilations': [1, 2, 4, 8]}


c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Saved best params to C:\capstone_project_unfc\model\experiments-pool\artifacts\best_tcn_params.parquet | MAE = 6.401808193751744


In [6]:
# Load best params from parquet (if exists) and train final global TCN
import ast
tcn_config = None
if BEST_TCN_PARAMS_PATH.exists():
    try:
        best_df = pd.read_parquet(BEST_TCN_PARAMS_PATH)
        if not best_df.empty:
            row = best_df.iloc[0]
            tcn_config = {
                "seq_len": int(row["seq_len"]),
                "epochs": int(row["epochs"]),
                "filters": int(row["filters"]),
                "kernel_size": int(row["kernel_size"]),
                "dilations": ast.literal_eval(row["dilations_str"]),
            }
            print("Loaded best TCN params from", BEST_TCN_PARAMS_PATH)
    except Exception:
        pass
global_tcn = train_global_tcn(stacked, FORECAST_HORIZON, config=tcn_config)
print("Global TCN trained on", len(build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)), "pooled train rows.")

Loaded best TCN params from C:\capstone_project_unfc\model\experiments-pool\artifacts\best_tcn_params.parquet


c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Global TCN trained on 11960 pooled train rows.


In [7]:
model_name = "tcn"
pred_tcn = run_backtest(stacked, global_tcn, FORECAST_HORIZON)
print(pred_tcn.groupby("symbol").size() if not pred_tcn.empty else "No predictions.")
pred_tcn.head()

symbol
AAPL     56
AMZN     56
GOOGL    56
JNJ      56
JPM      56
MSFT     56
NVDA     56
SPY      56
WMT      56
XOM      56
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-02,286.190002,283.804443,0,AAPL
1,2025-12-03,284.149994,284.327728,0,AAPL
2,2025-12-04,280.700012,284.795624,0,AAPL
3,2025-12-05,278.779999,285.354279,0,AAPL
4,2025-12-08,277.890015,285.910278,0,AAPL


In [8]:
if pred_tcn.empty:
    metrics_rows = [{"model": model_name, "symbol": sym, "MAE": np.nan, "RMSE": np.nan, "MAPE_%": np.nan} for sym in list(TICKERS)]
    metrics_rows.append({"model": model_name, "symbol": "overall", "MAE": np.nan, "RMSE": np.nan, "MAPE_%": np.nan})
else:
    metrics_rows = []
    for sym in pred_tcn["symbol"].unique():
        sub = pred_tcn[pred_tcn["symbol"] == sym]
        m = compute_metrics_averaged_over_windows(sub)
        metrics_rows.append({"model": model_name, "symbol": sym, **m})
    m_overall = compute_metrics_averaged_over_windows(pred_tcn)
    metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_tcn_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_tcn_pool.parquet")

   model   symbol        MAE       RMSE    MAPE_%
0    tcn     AAPL   7.469595   8.367934  2.832840
1    tcn     MSFT  12.755368  13.909257  2.889906
2    tcn    GOOGL   8.599055   9.714333  2.714584
3    tcn     AMZN   9.298197  10.235230  4.158367
4    tcn      JPM   7.072633   8.042576  2.264744
5    tcn      JNJ   3.657304   4.082180  1.658052
6    tcn      WMT   2.140097   2.463322  1.770356
7    tcn      SPY   7.351967   8.194759  1.074401
8    tcn      XOM   4.270257   4.597843  3.115399
9    tcn     NVDA   3.605327   4.284591  1.970933
10   tcn  overall   6.621980   8.918879  2.444958
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_tcn_pool.parquet
